<a href="https://colab.research.google.com/github/satyam72kr/India-Climate-Analysis/blob/main/Chest_X_Ray.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

print("🔄 Step 1: Preparing Medical Image Transforms...")
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# आपके गूगल ड्राइव 'data' फोल्डर का सटीक पाथ
train_dataset = datasets.ImageFolder(root='/content/drive/MyDrive/data/train', transform=transform)
val_dataset = datasets.ImageFolder(root='/content/drive/MyDrive/data/val', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print("🧠 Step 2: Loading Pre-trained ResNet18...")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print("🚀 Step 3: Starting Training Loop (3 Epochs)...")
model.train()
for epoch in range(3):
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    print(f"✅ Epoch {epoch+1}/3 Completed! Loss: {running_loss/len(train_loader):.4f}")

print("\n📊 Step 4: Final Evaluation Metrics...")
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("\n🔥 COPY THESE RESULTS FOR YOUR INTERVIEW 🔥")
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))


🔄 Step 1: Preparing Medical Image Transforms...
🧠 Step 2: Loading Pre-trained ResNet18...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 132MB/s]


🚀 Step 3: Starting Training Loop (3 Epochs)...
✅ Epoch 1/3 Completed! Loss: 0.1038
✅ Epoch 2/3 Completed! Loss: 0.0361
✅ Epoch 3/3 Completed! Loss: 0.0156

📊 Step 4: Final Evaluation Metrics...

🔥 COPY THESE RESULTS FOR YOUR INTERVIEW 🔥
              precision    recall  f1-score   support

      NORMAL       1.00      0.88      0.93         8
   PNEUMONIA       0.89      1.00      0.94         8

    accuracy                           0.94        16
   macro avg       0.94      0.94      0.94        16
weighted avg       0.94      0.94      0.94        16

